# ATR environment

Install Kraken/Ketos and fine-tune a shared recognition model on a selected PAGE XML ground truth.

Expected Google Drive layout:

```text
/content/drive/Shareddrives/kraken/workdir/
├── shared-models/
│   └── *.mlmodel
└── cwd/
    └── gtNNN/
        ├── images/
        ├── xmlpages/
        └── models/
```

Run the cells from top to bottom. The first code cell selects the ground truth, base model, script optimization, and optional output prefix.


## 1. Select the ground truth, base model, and script optimization

In [ ]:
from pathlib import Path
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output

drive.mount("/content/drive", force_remount=False)

WORKDIR_ROOT = Path("/content/drive/Shareddrives/kraken/workdir")
CWD_ROOT = WORKDIR_ROOT / "cwd"
SHARED_MODELS_DIR = WORKDIR_ROOT / "shared-models"

ground_truth_dirs = sorted(
    path for path in CWD_ROOT.glob("gt[0-9]*")
    if path.is_dir()
)
shared_models = sorted(SHARED_MODELS_DIR.glob("*.mlmodel"))

if not ground_truth_dirs:
    raise FileNotFoundError(f"No gtNNN directory found in {CWD_ROOT}")
if not shared_models:
    raise FileNotFoundError(f"No .mlmodel file found in {SHARED_MODELS_DIR}")

gt_dropdown = widgets.Dropdown(
    options=[(path.name, path) for path in ground_truth_dirs],
    description="Ground truth:",
    style={"description_width": "initial"},
)
model_dropdown = widgets.Dropdown(
    options=[(path.name, path) for path in shared_models],
    description="Base model:",
    style={"description_width": "initial"},
)
script_dropdown = widgets.Dropdown(
    options=[
        ("optimization for latin scripts", "latin"),
        ("optimization for arabic scripts", "arabic"),
    ],
    value="latin",
    description="Script optimization:",
    style={"description_width": "initial"},
)
output_prefix = widgets.Text(
    value=f"{shared_models[0].stem}_ft",
    description="Output prefix:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px"),
)
apply_button = widgets.Button(description="Apply configuration", button_style="primary")
configuration_output = widgets.Output()

def suggest_output_prefix(*_):
    output_prefix.value = f"{model_dropdown.value.stem}_ft"

model_dropdown.observe(suggest_output_prefix, names="value")

def apply_configuration(_):
    global GT_ROOT, IMAGES_DIR, PAGE_XML_DIR, MODELS_DIR
    global BASE_MODEL, OUTPUT_PREFIX, SCRIPT_OPTIMIZATION

    prefix = output_prefix.value.strip()
    if prefix and Path(prefix).name != prefix:
        raise ValueError("Enter a prefix, not a path.")
    if prefix.endswith(".mlmodel"):
        prefix = prefix[:-8]
        output_prefix.value = prefix

    GT_ROOT = Path(gt_dropdown.value)
    IMAGES_DIR = GT_ROOT / "images"
    PAGE_XML_DIR = GT_ROOT / "xmlpages"
    MODELS_DIR = GT_ROOT / "models"
    BASE_MODEL = Path(model_dropdown.value)
    SCRIPT_OPTIMIZATION = script_dropdown.value
    OUTPUT_PREFIX = prefix

    with configuration_output:
        clear_output()
        print("Configuration applied")
        print("Ground truth:", GT_ROOT)
        print("Images:     ", IMAGES_DIR)
        print("PAGE XML:   ", PAGE_XML_DIR)
        print("Base model: ", BASE_MODEL)
        print("Optimization:", SCRIPT_OPTIMIZATION)
        print("Output prefix:", OUTPUT_PREFIX or "(none)")

display(widgets.VBox([
    gt_dropdown,
    model_dropdown,
    script_dropdown,
    output_prefix,
    apply_button,
    configuration_output,
]))
apply_button.on_click(apply_configuration)
apply_configuration(None)

Mounted at /content/drive


## 2. Install and check Kraken/Ketos

In [ ]:
import platform
import shutil
import subprocess
import sys

def run(command, *, check=True, cwd=None):
    print("$", " ".join(str(part) for part in command))
    return subprocess.run(command, check=check, cwd=cwd)

print("Python:", sys.version)
print("Platform:", platform.platform())

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Kraken requires Python 3.10–3.13.")

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libpangocairo-1.0-0", "libxml2", "libxslt1.1"])
run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-U", "kraken[pdf]"])

print("kraken:", shutil.which("kraken"))
print("ketos:  ", shutil.which("ketos"))
run(["kraken", "--version"], check=False)
run(["ketos", "--version"], check=False)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as error:
    print("Torch check failed:", error)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
$ apt-get update -qq
$ apt-get install -y -qq libpangocairo-1.0-0 libxml2 libxslt1.1
$ /usr/bin/python3 -m pip install -U pip setuptools wheel
$ /usr/bin/python3 -m pip install -U kraken[pdf]
kraken: /usr/local/bin/kraken
ketos:   /usr/local/bin/ketos
$ kraken --version
$ ketos --version
Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3. Set training parameters

In [ ]:
# Dataset split
VALIDATION_RATIO = 0.10
RANDOM_SEED = 42

# Training
EPOCHS = 30
MIN_EPOCHS = 5
EARLY_STOPPING_LAG = 8
BATCH_SIZE = 8
NUM_WORKERS = 2
NORMALIZATION = "NFD"       # Applied consistently to training and evaluation text.
RESIZE_MODE = "union"       # Preserve the base codec and add new characters.
USE_AUGMENTATION = True
PRECISION = "32-true"

# Temporary local workspace. It is rebuilt on every run.
LOCAL_TRAINING_ROOT = Path("/content/atr_environment_work") / GT_ROOT.name
LOCAL_MODEL_OUTPUT_DIR = LOCAL_TRAINING_ROOT / "training"

## 4. Validate and prepare the PAGE XML dataset

In [ ]:
import os
import random
import shutil
import xml.etree.ElementTree as ET

for required_path in (GT_ROOT, IMAGES_DIR, PAGE_XML_DIR, BASE_MODEL):
    if not required_path.exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_TRAINING_ROOT.exists():
    shutil.rmtree(LOCAL_TRAINING_ROOT)
LOCAL_TRAINING_ROOT.mkdir(parents=True)

def find_image(page_xml_path):
    root = ET.parse(page_xml_path).getroot()
    page = root.find(".//{*}Page")
    image_filename = page.get("imageFilename") if page is not None else None

    candidates = []
    if image_filename:
        candidates.append(IMAGES_DIR / Path(image_filename).name)
    for extension in (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".jp2"):
        candidates.extend([
            IMAGES_DIR / f"{page_xml_path.stem}{extension}",
            IMAGES_DIR / f"{page_xml_path.stem}{extension.upper()}",
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No matching image found for {page_xml_path.name}")

def count_transcribed_lines(page_xml_path):
    root = ET.parse(page_xml_path).getroot()
    return sum(
        1 for line in root.findall(".//{*}TextLine")
        if (unicode_element := line.find("./{*}TextEquiv/{*}Unicode")) is not None
        and unicode_element.text
        and unicode_element.text.strip()
    )

def prepare_pair(page_xml_source, image_source):
    page_xml_target = LOCAL_TRAINING_ROOT / page_xml_source.name
    image_target = LOCAL_TRAINING_ROOT / image_source.name

    tree = ET.parse(page_xml_source)
    page = tree.getroot().find(".//{*}Page")
    if page is None:
        raise ValueError(f"No <Page> element in {page_xml_source.name}")
    page.set("imageFilename", image_target.name)
    tree.write(page_xml_target, encoding="UTF-8", xml_declaration=True)

    try:
        os.symlink(image_source, image_target)
    except OSError:
        shutil.copy2(image_source, image_target)
    return page_xml_target

page_xml_files = sorted(PAGE_XML_DIR.glob("*.xml"))
if len(page_xml_files) < 2:
    raise RuntimeError(f"At least two PAGE XML files are required; found {len(page_xml_files)}.")

prepared_xml_files = []
total_lines = 0
for page_xml_source in page_xml_files:
    image_source = find_image(page_xml_source)
    line_count = count_transcribed_lines(page_xml_source)
    if line_count == 0:
        raise RuntimeError(f"No transcribed TextLine found in {page_xml_source.name}")
    prepared_xml_files.append(prepare_pair(page_xml_source, image_source))
    total_lines += line_count

random.Random(RANDOM_SEED).shuffle(prepared_xml_files)
validation_count = max(1, round(len(prepared_xml_files) * VALIDATION_RATIO))
validation_xml_files = sorted(prepared_xml_files[:validation_count])
training_xml_files = sorted(prepared_xml_files[validation_count:])

TRAINING_LIST = LOCAL_TRAINING_ROOT / "train.lst"
VALIDATION_LIST = LOCAL_TRAINING_ROOT / "validation.lst"
TRAINING_LIST.write_text("\n".join(map(str, training_xml_files)) + "\n", encoding="utf-8")
VALIDATION_LIST.write_text("\n".join(map(str, validation_xml_files)) + "\n", encoding="utf-8")

print("PAGE XML files:", len(prepared_xml_files))
print("Transcribed lines:", total_lines)
print("Training files:", len(training_xml_files))
print("Validation files:", len(validation_xml_files))

PAGE XML files: 23
Transcribed lines: 550
Training files: 21
Validation files: 2


## 5. Build the Ketos command

In [ ]:
def command_help(command):
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    return result.stdout + "\n" + result.stderr

ketos_help = command_help(["ketos", "--help"])
training_help = command_help(["ketos", "train", "--help"])

device = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        device = "cuda:0"
except Exception:
    pass

global_arguments = []
training_arguments = []

if "--device" in ketos_help:
    global_arguments += ["--device", device]
elif "--device" in training_help:
    training_arguments += ["--device", device]
elif "-d," in training_help:
    training_arguments += ["-d", device]

if "--precision" in ketos_help:
    global_arguments += ["--precision", PRECISION]
elif "--precision" in training_help:
    training_arguments += ["--precision", "32"]

training_arguments += ["--format-type", "xml"] if "--format-type" in training_help else ["-f", "xml"]
training_arguments += ["--load", str(BASE_MODEL)] if "--load" in training_help else ["-i", str(BASE_MODEL)]
training_arguments += ["--output", str(LOCAL_MODEL_OUTPUT_DIR)] if "--output" in training_help else ["-o", str(LOCAL_MODEL_OUTPUT_DIR)]

if "--training-data" in training_help:
    training_arguments += ["--training-data", str(TRAINING_LIST)]
elif "--training-files" in training_help:
    training_arguments += ["--training-files", str(TRAINING_LIST)]
else:
    training_arguments += ["-t", str(TRAINING_LIST)]

if "--evaluation-data" in training_help:
    training_arguments += ["--evaluation-data", str(VALIDATION_LIST)]
elif "--evaluation-files" in training_help:
    training_arguments += ["--evaluation-files", str(VALIDATION_LIST)]
else:
    training_arguments += ["-e", str(VALIDATION_LIST)]

optional_arguments = [
    ("--resize", RESIZE_MODE),
    ("--epochs", str(EPOCHS)),
    ("--min-epochs", str(MIN_EPOCHS)),
    ("--lag", str(EARLY_STOPPING_LAG)),
    ("--batch-size", str(BATCH_SIZE)),
]
if NORMALIZATION:
    optional_arguments.append(("--normalization", NORMALIZATION))
for option, value in optional_arguments:
    if option in training_help:
        training_arguments += [option, value]

# Script-specific text handling. Arabic PAGE XML transcriptions are expected
# in logical Unicode order; Ketos reorders them for the recognition network.
if SCRIPT_OPTIMIZATION == "arabic":
    if "--reorder" in training_help:
        training_arguments.append("--reorder")
    if "--base-dir" in training_help:
        training_arguments += ["--base-dir", "R"]

# Normalize Unicode whitespace to a plain space when supported.
if "--normalize-whitespace" in training_help:
    training_arguments.append("--normalize-whitespace")

if "--num-workers" in ketos_help:
    global_arguments += ["--num-workers", str(NUM_WORKERS)]
elif "--num-workers" in training_help:
    training_arguments += ["--num-workers", str(NUM_WORKERS)]
elif "--workers" in training_help:
    training_arguments += ["--workers", str(NUM_WORKERS)]

if USE_AUGMENTATION and "--augment" in training_help:
    training_arguments.append("--augment")
if "--weights-format" in training_help:
    training_arguments += ["--weights-format", "coreml"]

KETOS_COMMAND = ["ketos", *global_arguments, "train", *training_arguments]
print("Device:", device)
print("Script optimization:", SCRIPT_OPTIMIZATION)
print("Command:")
print(" ".join(KETOS_COMMAND))

Device: cuda:0
Script optimization: arabic
Command:
ketos --device cuda:0 --precision 32-true train --format-type xml --load /content/drive/Shareddrives/kraken/workdir/shared-models/model-litho_best.mlmodel --output /content/atr_environment_work/gt003/training --training-data /content/atr_environment_work/gt003/train.lst --evaluation-data /content/atr_environment_work/gt003/validation.lst --resize union --epochs 30 --min-epochs 5 --lag 8 --batch-size 8 --normalization NFD --reorder --base-dir R --normalize-whitespace --augment --weights-format coreml


## 6. Fine-tune and save the model

In [ ]:
run([sys.executable, "-m", "pip", "install", "-U", "transformers"])


$ /usr/bin/python3 -m pip install -U transformers


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-U', 'transformers'], returncode=0)

In [ ]:
import time

os.environ.setdefault("WANDB_DISABLED", "true")
LOCAL_MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_started = time.time()
run(KETOS_COMMAND, cwd=LOCAL_MODEL_OUTPUT_DIR)

new_models = [
    path for path in LOCAL_MODEL_OUTPUT_DIR.rglob("*.mlmodel")
    if path.stat().st_mtime >= training_started - 5
]
if not new_models:
    raise RuntimeError(
        "Training finished, but no new .mlmodel file was found in the volatile output directory."
    )

best_candidates = [path for path in new_models if path.name.startswith("best_")]
produced_model = max(best_candidates or new_models, key=lambda path: path.stat().st_mtime)

final_filename = f"{OUTPUT_PREFIX}-{produced_model.name}" if OUTPUT_PREFIX else produced_model.name
FINAL_MODEL = MODELS_DIR / final_filename
MODELS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(produced_model, FINAL_MODEL)

print("Fine-tuning completed.")
print("Temporary Ketos model:", produced_model)
print("Final model saved to Google Drive:", FINAL_MODEL)
print("Temporary checkpoints:", LOCAL_MODEL_OUTPUT_DIR)

$ ketos --device cuda:0 --precision 32-true train --format-type xml --load /content/drive/Shareddrives/kraken/workdir/shared-models/model-litho_best.mlmodel --output /content/atr_environment_work/gt003/training --training-data /content/atr_environment_work/gt003/train.lst --evaluation-data /content/atr_environment_work/gt003/validation.lst --resize union --epochs 30 --min-epochs 5 --lag 8 --batch-size 8 --normalization NFD --reorder --base-dir R --normalize-whitespace --augment --weights-format coreml
Fine-tuning completed.
Temporary Ketos model: /content/atr_environment_work/gt003/training/best_0.7574.mlmodel
Final model saved to Google Drive: /content/drive/Shareddrives/kraken/workdir/cwd/gt003/models/model-litho_best_ft-best_0.7574.mlmodel
Temporary checkpoints: /content/atr_environment_work/gt003/training


## 7. OPTIONAL: augmented data

In [13]:
from pathlib import Path
import tarfile

destination = Path('/content/atr_environment_work')
destination.mkdir(parents=True, exist_ok=True)
with tarfile.open('/gt003_aggressive.tar.bz2', 'r:bz2') as archive:
    archive.extractall(destination, filter='data')


In [14]:
!python /content/atr_environment_work/gt003_aggressive/scripts/run_training.py --mode augmented --run

Streaming output truncated to the last 5000 lines.
                               0:01:20               33.895        5/8 0.77546  
                                                     val_accuracy:              
                                                     0.760                      
                                                     val_word_acc…              
                                                     0.209                      
                                                     train_loss_e…              
stage 32/∞ ━━━╸━━━━━━━━ 75/249 0:00:34 •    2.19it/s train_loss_s… early_stoppi…
                               0:01:20               30.149        5/8 0.77546  
                                                     val_accuracy:              
                                                     0.760                      
                                                     val_word_acc…              
                                                     0.209